In [30]:
from local_pool_price import *
import plotly.express as px
import plotly.io
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mainnet_launch.data_fetching.get_state_by_block import _build_blocks_to_use_dont_clip
from mainnet_launch.constants import WORKING_DATA_DIR


plotly.io.templates.default = None

In [49]:
sDAI_backing = Call(
    sDAI,
    ["convertToAssets(uint256)(uint256)", int(1e18)],
    [("sDAI_backing", safe_normalize_with_bool_success)],
)

sUSDs_backing = Call(
    sUSDs,
    ["convertToAssets(uint256)(uint256)", int(1e18)],
    [("sUSDs_backing", safe_normalize_with_bool_success)],
)

from decimal import Decimal, localcontext


def dsr_to_apr_value(success, dsr: int) -> float:
    if success:
        # Use a high precision context
        with localcontext() as ctx:
            ctx.prec = 100  # Adjust as needed for precision
            # Convert dsr from its ray (27-digit) representation to a Decimal
            r = Decimal(dsr) / Decimal("1e27")
            # Since r is very close to 1, set x = r - 1
            x = r - Decimal(1)
            # There are 31,536,000 seconds in a year
            seconds = Decimal(31536000)
            # Compute APR using the exponential of (seconds * x)
            apr_decimal = Decimal(100) * ((seconds * x).exp() - Decimal(1))
            return float(apr_decimal)
    else:
        return 0.0


MCD_pot_address = "0x197E90f9FAD81970bA7976f33CbD77088E5D7cf7"
dsr_call = Call(
    MCD_pot_address,
    ["dsr()(uint256)"],
    [("dsr", dsr_to_apr_value)],
)

USDS_address = "0xA3931D71877C0E7A3148CB7EB4463524FEC27FBD"



srr_call = Call(
    USDS_address,
    ["ssr()(uint256)"],
    [("ssr", dsr_to_apr_value)],
)


calls = [sDAI_backing, sUSDs_backing, dsr_call, srr_call]

blocks = _build_blocks_to_use_dont_clip(ETH_CHAIN, 18_000_000, approx_num_blocks_per_day=24)

df = get_raw_state_by_blocks(calls, blocks, ETH_CHAIN, include_block_number=True)
df

,sDAI_backing,sUSDs_backing,dsr,ssr,block
timestamp,,,,,
2023-08-26 16:21:35+00:00,1.031763,NaN,5.0,0.0,18000000
2023-08-26 17:21:47+00:00,1.031769,NaN,5.0,0.0,18000300
2023-08-26 18:21:47+00:00,1.031774,NaN,5.0,0.0,18000600
2023-08-26 19:22:23+00:00,1.031780,NaN,5.0,0.0,18000900
2023-08-26 20:22:47+00:00,1.031786,NaN,5.0,0.0,18001200
...,...,...,...,...,...
2025-03-30 19:45:11+00:00,1.153372,1.045696,3.5,4.5,22161900
2025-03-30 20:45:11+00:00,1.153376,1.045702,3.5,4.5,22162200
2025-03-30 21:45:11+00:00,1.153381,1.045707,3.5,4.5,22162500


In [ ]:
# raw_df = df.copy()
# df = raw_df.copy()

In [92]:
df = df.resample("1d").last()
n_days = 14
df["sDAI_backing_14_days_ago"] = df["sDAI_backing"].shift(n_days)


def _compute_sDAI_expected_apy_from_14_day_method(row: dict) -> float:
    start = row["sDAI_backing_14_days_ago"]
    end   = row["sDAI_backing"]

    if pd.isna(start) or start <= 0:
        return float("nan")

    # ratio = end / start over the 14-day window
    ratio = end / start

    # annualize with compounding
    #  ratio^(365/14) - 1
    implied_apy = (ratio ** (365.0 / 14.0)) - 1

    return implied_apy * 100.0


df["sDAI_14_day_apy"] = df.apply(lambda row: _compute_sDAI_expected_apy_from_14_day_method(row), axis=1)


def _compute_expected_forward_dai(row: dict):
    current = row["sDAI_backing"]
    factor = 1 + (row["sDAI_14_day_apy"] / 100) * (n_days / 365)
    return current * factor


df["expected_after_14_day_sDAI_backing"] = df.apply(lambda row: _compute_expected_forward_dai(row), axis=1)
df["actual_after_14_day_sDAI_backing"] = df["sDAI_backing"].shift(-n_days)


def _compute_actualied_apy_from_t_0_to_t14(row: dict):
    start = row["sDAI_backing"]
    end   = row["actual_after_14_day_sDAI_backing"]

    if pd.isna(start) or start <= 0:
        return float("nan")

    # ratio = end / start over the 14-day window
    ratio = end / start

    # annualize with compounding
    #  ratio^(365/14) - 1
    implied_apy = (ratio ** (365.0 / 14.0)) - 1

    return implied_apy * 100.0



df["actualized_apy_over_14_days"] = df.apply(lambda row: _compute_actualied_apy_from_t_0_to_t14(row), axis=1)


df.columns

px.line(df[['actualized_apy_over_14_days', 'dsr', 'sDAI_14_day_apy']], title='Actual 14 day APY vs DSR vs 14 day Backwards Looking APY')

In [93]:
df['dsr_error'] =  df['dsr'] - df['actualized_apy_over_14_days']
df['14_day_apy_error'] = df['sDAI_14_day_apy'] - df['actualized_apy_over_14_days']
# DSR includes compoundings

px.area(df[['dsr_error', '14_day_apy_error' ]].abs(), title='Absolute Error Projected - Actual')

In [89]:
px.line(df[['dsr_error', '14_day_apy_error' ]].abs(), title='Absolute Error Projected - Actual')

In [116]:
(df['DSR_error'].abs() <= df['14_day_apy_error'].abs()).value_counts()

True     407
False    176
Name: count, dtype: int64

In [121]:
df['DSR_error - 14 day error'] = df['DSR_error'].abs() - df['14_day_apy_error'].abs()

In [ ]:
px.line(df['DSR_error - 14 day error'], 'How much the')

In [124]:

df[['DSR_error', '14_day_apy_error']].describe().round(2)

,DSR_error,14_day_apy_error
count,569.00,555.00
mean,0.01,-0.00
std,1.09,1.66
min,-9.38,-9.95
25%,-0.01,-0.01
50%,0.00,0.01
75%,0.01,0.42
max,3.81,3.96


In [95]:
px.scatter(df, x= 'actualized_apy_over_14_days', y='dsr', title='Acutal APY vs DSR')

In [ ]:
px.ecdf(df[['DSR_error', '14_day_apy_error']], title='14-day method vs DSR method error')

In [107]:
px.ecdf(df[['DSR_error', '14_day_apy_error']].abs(), title='14-day method vs DSR method error')

In [129]:
df[['DSR_error', '14_day_apy_error']].abs().describe(percentiles=[.8,.9, .95]).round(2)

,DSR_error,14_day_apy_error
count,569.00,555.00
mean,0.40,0.80
std,1.02,1.45
min,0.00,0.00
50%,0.01,0.08
80%,0.50,1.47
90%,1.34,2.49
95%,2.03,3.32
max,9.38,9.95


In [131]:
df[['DSR_error', '14_day_apy_error']].describe(percentiles=[.8,.9, .95]).round(2)

,DSR_error,14_day_apy_error
count,569.00,555.00
mean,0.01,-0.00
std,1.09,1.66
min,-9.38,-9.95
50%,0.00,0.01
80%,0.11,0.67
90%,0.75,1.50
95%,1.46,2.27
max,3.81,3.96


In [126]:
!poetry add tabulate

15137.05s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


The following packages are already present in the pyproject.toml and will be skipped:

  - tabulate

If you want to update it to the latest compatible version, you can use `poetry update package`.
If you prefer to upgrade it to the latest available version, you can use `poetry add package@latest`.

Nothing to add.


In [83]:
px.violin(df[['DSR_error', '14_day_apy_error']], title='14-day vs dsr absolute error')

In [96]:
px.scatter(df, 'actualized_apy_over_14_days', 'sDAI_14_day_apy', title='Actual APY vs 14 day trailing')

In [52]:
df.columns

Index(['sDAI_backing', 'sUSDs_backing', 'dsr', 'ssr', 'block',
       'sDAI_backing_14_days_ago', 'sDAI_14_day_apr',
       'expected_after_14_day_sDAI_backing',
       'actual_after_14_day_sDAI_backing', 'actualized_apr_over_14_days'],
      dtype='object')

In [ ]:
df['dsr_error'] = df['s']

In [ ]:
break

In [ ]:
df = df.resample("1d").last()
df["sDAI_prior"] = df["sDAI_backing"].shift(14)
df["sUSDs_prior"] = df["sUSDs_backing"].shift(14)


def _compute_dai_apr(row: dict) -> float:
    diff = row["sDAI_backing"] - row["sDAI_prior"]
    return 100 * (diff * 365) / 14


def _compute_sUSDs_apr(row: dict) -> float:
    diff = row["sUSDs_backing"] - row["sUSDs_prior"]
    return 100 * (diff * 365) / 14


df["sDAI_14_day_apr"] = df.apply(lambda row: _compute_dai_apr(row), axis=1)
df["sUSDs_14_day_apr"] = df.apply(lambda row: _compute_sUSDs_apr(row), axis=1)


def _compute_expected_forward_dai(row: dict):
    current = row["sDAI_backing"]
    # Calculate the factor for 14 days at a 5% annual rate.
    factor = 1 + (row["sDAI_14_day_apr"] / 100) * (14 / 365)
    return current * factor


def _compute_expected_forward_sUSDs(row: dict):
    current = row["sUSDs_backing"]
    # Calculate the factor for 14 days at a 5% annual rate.
    factor = 1 + (row["sUSDs_14_day_apr"] / 100) * (14 / 365)
    return current * factor


df["expected_after_14_day_sDAI_backing"] = df.apply(lambda row: _compute_expected_forward_dai(row), axis=1)
df["expected_after_14_day_sUSDs_backing"] = df.apply(lambda row: _compute_expected_forward_sUSDs(row), axis=1)

df["actual_after_14_day_sDAI_backing"] = df["sDAI_backing"].shift(-14)
df["actual_after_14_day_sUSDs_backing"] = df["sUSDs_backing"].shift(-14)


px.line(df)